# Sprint 10: OilyGiant (escolha da melhor região para poços de petróleo)

A OilyGiant quer abrir 200 novos poços de petróleo e precisa decidir em qual das três regiões investir. Vou treinar uma regressão linear para prever o volume de reservas de cada poço, selecionar os 200 melhores de cada região e usar bootstrapping para estimar o lucro e o risco de prejuízo.

Condições do negócio:
- Orçamento de 100 milhões de dólares para 200 poços
- Cada unidade de produto (mil barris) rende 4.500 dólares
- São estudados 500 pontos por região e escolhidos os 200 melhores
- Só considerar regiões com risco de prejuízo abaixo de 2.5%

## 1. Importações e carregamento dos dados

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [2]:
geo_0 = pd.read_csv('/datasets/geo_data_0.csv')
geo_1 = pd.read_csv('/datasets/geo_data_1.csv')
geo_2 = pd.read_csv('/datasets/geo_data_2.csv')

In [3]:
geo_0.head()

,id,f0,f1,f2,product
0,txEyH,0.705745,-0.497823,1.221170,105.280062
1,2acmU,1.334711,-0.340164,4.365080,73.037750
2,409Wp,1.022732,0.151990,1.419926,85.265647
3,iJLyR,-0.032172,0.139033,2.978566,168.620776
4,Xdl7t,1.988431,0.155413,4.751769,154.036647


In [4]:
geo_0.info()
print()
print('Nulos região 0:', geo_0.isnull().sum().sum())
print('Nulos região 1:', geo_1.isnull().sum().sum())
print('Nulos região 2:', geo_2.isnull().sum().sum())
print()
print('Duplicados região 0:', geo_0['id'].duplicated().sum())
print('Duplicados região 1:', geo_1['id'].duplicated().sum())
print('Duplicados região 2:', geo_2['id'].duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB

Nulos região 0: 0
Nulos região 1: 0
Nulos região 2: 0

Duplicados região 0: 10
Duplicados região 1: 4
Duplicados região 2: 4


In [5]:
geo_0.describe()

,f0,f1,f2,product
count,100000.000000,100000.000000,100000.000000,100000.000000
mean,0.500419,0.250143,2.502647,92.500000
std,0.871832,0.504433,3.248248,44.288691
min,-1.408605,-0.848218,-12.088328,0.000000
25%,-0.072580,-0.200881,0.287748,56.497507
50%,0.502360,0.250252,2.515969,91.849972
75%,1.073581,0.700646,4.715088,128.564089
max,2.362331,1.343769,16.003790,185.364347


Os três datasets têm 100.000 linhas e as mesmas colunas: id, f0, f1, f2 e product. Não há valores ausentes. Existem alguns ids duplicados (10 na região 0, 4 nas regiões 1 e 2), mas são poucos diante de 100 mil linhas e provavelmente são apenas repetição do código identificador, não linhas inteiras iguais, então não vão atrapalhar o modelo. A coluna id não entra no treino. As features f0, f1 e f2 são numéricas e o alvo é product.

## 2. Treino e teste do modelo por região

Para cada região divido os dados em treino e validação (75:25), escalo as features, treino a regressão linear e calculo o volume médio previsto e o REQM. Coloquei tudo numa função para não repetir código nas três regiões.

In [14]:
def treinar_avaliar(df, nome):
    features = df.drop(['id', 'product'], axis=1)
    target = df['product']

    features_train, features_valid, target_train, target_valid = train_test_split(
        features, target, test_size=0.25, random_state=12345
    )

    scaler = StandardScaler()
    scaler.fit(features_train)
    features_train = scaler.transform(features_train)
    features_valid = scaler.transform(features_valid)

    model = LinearRegression()
    model.fit(features_train, target_train)
    predicted_valid = model.predict(features_valid)

    target_valid = target_valid.reset_index(drop=True)
    predicted_valid = pd.Series(predicted_valid)

    rmse = mean_squared_error(target_valid, predicted_valid) ** 0.5
    media = predicted_valid.mean()

    print(f'--- {nome} ---')
    print(f'Volume médio previsto: {media:.2f} mil barris')
    print(f'REQM: {rmse:.2f}')
    print()

    return target_valid, predicted_valid

In [7]:
target_valid_0, predicted_0 = treinar_avaliar(geo_0, 'Região 0')
target_valid_1, predicted_1 = treinar_avaliar(geo_1, 'Região 1')
target_valid_2, predicted_2 = treinar_avaliar(geo_2, 'Região 2')

--- Região 0 ---
Volume médio previsto: 92.59 mil barris
REQM: 37.58

--- Região 1 ---
Volume médio previsto: 68.73 mil barris
REQM: 0.89

--- Região 2 ---
Volume médio previsto: 94.97 mil barris
REQM: 40.03



As regiões 0 e 2 têm volume médio previsto mais alto (entre 92 e 95 mil barris), mas o REQM é grande (cerca de 38 a 40), o que indica previsões com bastante erro. Já a região 1 tem volume médio menor (cerca de 69 mil barris), mas o REQM é muito baixo (0.89). Ou seja, o modelo acerta quase em cheio nessa região, provavelmente porque os dados dela têm relação quase linear com o alvo. Esse REQM baixo deixa a seleção dos melhores poços bem mais confiável na região 1.

## 3. Preparação para o cálculo de lucro

Vou armazenar as constantes do negócio e calcular o ponto de equilíbrio, ou seja, quanto cada poço precisa produzir em média para não dar prejuízo.

In [8]:
ORCAMENTO = 100_000_000
N_POCOS = 200
N_ESTUDADOS = 500
RECEITA_UNIDADE = 4500

equilibrio = ORCAMENTO / N_POCOS / RECEITA_UNIDADE
print(f'Ponto de equilíbrio por poço: {equilibrio:.1f} mil barris')
print()
print(f'Volume médio real região 0: {target_valid_0.mean():.1f}')
print(f'Volume médio real região 1: {target_valid_1.mean():.1f}')
print(f'Volume médio real região 2: {target_valid_2.mean():.1f}')

Ponto de equilíbrio por poço: 111.1 mil barris

Volume médio real região 0: 92.1
Volume médio real região 1: 68.7
Volume médio real região 2: 94.9


Cada poço precisa produzir em média 111.1 mil barris para os 200 poços juntos cobrirem o investimento. O volume médio das três regiões (entre 69 e 95 mil barris) fica abaixo desse ponto de equilíbrio, mas isso faz sentido porque não vamos abrir poços aleatórios e sim escolher os 200 melhores de cada região, que produzem bem acima da média.

## 4. Função de cálculo de lucro

A função pega os 200 poços com maior volume previsto, soma o volume real correspondente e calcula o lucro descontando o investimento.

In [9]:
def calcular_lucro(target, predicted, n_pocos=N_POCOS):
    indices_top = predicted.sort_values(ascending=False).index[:n_pocos]
    volume_selecionado = target[indices_top].sum()
    lucro = volume_selecionado * RECEITA_UNIDADE - ORCAMENTO
    return lucro

In [10]:
lucro_0 = calcular_lucro(target_valid_0, predicted_0)
lucro_1 = calcular_lucro(target_valid_1, predicted_1)
lucro_2 = calcular_lucro(target_valid_2, predicted_2)

print(f'Lucro região 0 (200 melhores): {lucro_0:,.2f} dólares')
print(f'Lucro região 1 (200 melhores): {lucro_1:,.2f} dólares')
print(f'Lucro região 2 (200 melhores): {lucro_2:,.2f} dólares')

Lucro região 0 (200 melhores): 33,208,260.43 dólares
Lucro região 1 (200 melhores): 24,150,866.97 dólares
Lucro região 2 (200 melhores): 27,103,499.64 dólares


Olhando só os 200 melhores poços, todas as regiões dão lucro positivo. Mas esse número é otimista porque assume que conseguimos escolher exatamente os melhores poços, com esse resultado a região 0 parece ser a mais promissora. Na prática só estudamos 500 pontos por vez e há incerteza nas previsões, então preciso do bootstrapping para medir o risco real.

## 5. Cálculo de risco e lucro com Bootstrapping

Para cada região, vou repetir 1.000 vezes o seguinte: sortear 500 poços (com reposição), escolher os 200 com maior volume previsto e calcular o lucro. Com a distribuição dos 1.000 lucros, calculo o lucro médio, o intervalo de confiança de 95% e o risco de prejuízo.

In [ ]:
def bootstrap_lucro(target, predicted, nome):
    state = np.random.RandomState(12345)
    lucros = []

    for i in range(1000):
        target_subsample = target.sample(n=N_ESTUDADOS, replace=True, random_state=state)
        predicted_subsample = predicted[target_subsample.index]
        lucros.append(calcular_lucro(target_subsample, predicted_subsample))

    lucros = pd.Series(lucros)

    lucro_medio = lucros.mean()
    lower = lucros.quantile(0.025)
    upper = lucros.quantile(0.975)
    risco = (lucros < 0).mean() * 100

    print(f'--- {nome} ---')
    print(f'Lucro médio: {lucro_medio:,.2f} dólares')
    print(f'Intervalo de confiança 95%: {lower:,.2f} a {upper:,.2f}')
    print(f'Risco de prejuízo: {risco:.1f}%')
    print()

    return lucro_medio, risco

In [12]:
media_0, risco_0 = bootstrap_lucro(target_valid_0, predicted_0, 'Região 0')
media_1, risco_1 = bootstrap_lucro(target_valid_1, predicted_1, 'Região 1')
media_2, risco_2 = bootstrap_lucro(target_valid_2, predicted_2, 'Região 2')

--- Região 0 ---
Lucro médio: 6,007,352.44 dólares
Intervalo de confiança 95%: 129,483.31 a 12,311,636.06
Risco de prejuízo: 2.0%

--- Região 1 ---
Lucro médio: 6,652,410.58 dólares
Intervalo de confiança 95%: 1,579,884.81 a 11,976,415.87
Risco de prejuízo: 0.3%

--- Região 2 ---
Lucro médio: 6,155,597.23 dólares
Intervalo de confiança 95%: -122,184.95 a 12,306,444.74
Risco de prejuízo: 3.0%



In [13]:
resumo = pd.DataFrame({
    'Região': ['Região 0', 'Região 1', 'Região 2'],
    'Lucro médio': [media_0, media_1, media_2],
    'Risco de prejuízo (%)': [risco_0, risco_1, risco_2]
})
print(resumo.to_string(index=False))

  Região  Lucro médio  Risco de prejuízo (%)
Região 0 6.007352e+06                    2.0
Região 1 6.652411e+06                    0.3
Região 2 6.155597e+06                    3.0


## 6. Conclusão

O critério do negócio diz para manter só as regiões com risco de prejuízo abaixo de 2.5%. Pelos resultados do bootstrapping, a região 0 ficou em 2.0%, a região 1 em 0.3% e a região 2 em 3.0%. A região 2 já sai de cena por estourar o limite de risco.

Entre as que passaram (0 e 1), a região 1 tem o maior lucro médio (cerca de 6.65 milhões contra 6.0 milhões da região 0) e o menor risco de longe (0.3%). O intervalo de confiança dela também é todo positivo, enquanto o da região 0 chega bem perto de zero no limite inferior.

Esse resultado faz sentido com a modelagem: a região 1 tinha REQM de 0.89, muito menor que as outras, então as previsões dela são bem mais confiáveis e dá para escolher os melhores poços com segurança mesmo o volume médio bruto sendo menor.

No passo 4 todas as regiões pareciam boas porque olhamos só os 200 melhores poços, mas a análise de risco mostra que só a região 1 junta lucro alto e risco baixo. Minha escolha é a região 1, e ela bate com as duas análises.